Downlond 100k

In [1]:
import os, zipfile, urllib.request
from pathlib import Path

DATA_DIR = Path("data_movielens_100k")
DATA_DIR.mkdir(parents=True, exist_ok=True)

URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
zip_path = DATA_DIR / "ml-100k.zip"

if not zip_path.exists():
    print("Downloading MovieLens 100K...")
    urllib.request.urlretrieve(URL, zip_path.as_posix())
else:
    print("Zip already downloaded:", zip_path)

# Unzip
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(DATA_DIR)

print("Extracted to:", DATA_DIR)
print("Top-level contents:", list(DATA_DIR.iterdir())[:5])

Zip already downloaded: data_movielens_100k\ml-100k.zip
Extracted to: data_movielens_100k
Top-level contents: [WindowsPath('data_movielens_100k/ml-100k'), WindowsPath('data_movielens_100k/ml-100k.zip')]


Load ratings (u.data)

In [2]:
import pandas as pd

ratings_path = DATA_DIR / "ml-100k" / "u.data"
# u.data format: user_id \t item_id \t rating \t timestamp
ratings = pd.read_csv(
    ratings_path,
    sep="\t",
    header=None,
    names=["user_id", "item_id", "rating", "timestamp"],
)

ratings.head(), ratings.shape

(   user_id  item_id  rating  timestamp
 0      196      242       3  881250949
 1      186      302       3  891717742
 2       22      377       1  878887116
 3      244       51       2  880606923
 4      166      346       1  886397596,
 (100000, 4))

Map raw IDs to contiguous indices

In [3]:
# Create contiguous indices for users and items
user_ids = ratings["user_id"].unique()
item_ids = ratings["item_id"].unique()

user2idx = {u:i for i,u in enumerate(sorted(user_ids))}
item2idx = {m:i for i,m in enumerate(sorted(item_ids))}

ratings["u"] = ratings["user_id"].map(user2idx).astype(int)
ratings["i"] = ratings["item_id"].map(item2idx).astype(int)

n_users = len(user2idx)
n_items = len(item2idx)

print("n_users:", n_users, "n_items:", n_items)
ratings[["u","i","rating"]].head()

n_users: 943 n_items: 1682


,u,i,rating
0,195,241,3
1,185,301,3
2,21,376,1
3,243,50,2
4,165,345,1


Train/test split (random 80/20 over observed ratings)

In [4]:

import numpy as np


rng = np.random.default_rng(42)
idx = np.arange(len(ratings))
rng.shuffle(idx)

split = int(0.8 * len(idx))
train_idx = idx[:split]
test_idx  = idx[split:]

train = ratings.iloc[train_idx].reset_index(drop=True)
test  = ratings.iloc[test_idx].reset_index(drop=True)

print("Train size:", len(train), "Test size:", len(test))

# Convert to numpy arrays for fast loops
train_u = train["u"].to_numpy(np.int64)
train_i = train["i"].to_numpy(np.int64)
train_r = train["rating"].to_numpy(np.float32)

test_u = test["u"].to_numpy(np.int64)
test_i = test["i"].to_numpy(np.int64)
test_r = test["rating"].to_numpy(np.float32)

Train size: 80000 Test size: 20000


Candidate Generator

In [5]:
import random

M = 10000
all_users = set(train['user_id'].unique())

user = random.choice(list(all_users))

def get_seen_movies(user_id, ratings):
    return set(ratings[ratings['user_id'] == user_id]['item_id'])

def generate_candidates(user, ratings):
    all_movies = set(ratings['item_id'].unique())
    seen_movies = get_seen_movies(user, ratings)
    
    return list(all_movies - seen_movies)

movies_unreview = generate_candidates(user, train)

def topM(ratings, user, M):
    candidates = generate_candidates(user, ratings)
    return candidates[:M]

movies_unreview = topM(train, user, M)

print(movies_unreview)

[2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 26, 27, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 52, 53, 57, 59, 60, 61, 62, 63, 65, 66, 67, 70, 72, 74, 75, 76, 77, 78, 80, 83, 84, 85, 86, 87, 88, 89, 90, 92, 93, 94, 95, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 135, 136, 137, 138, 139, 140, 141, 142, 145, 146, 147, 148, 149, 151, 152, 153, 154, 155, 157, 158, 160, 162, 163, 165, 166, 167, 169, 170, 171, 172, 175, 177, 178, 179, 180, 183, 184, 185, 187, 188, 189, 190, 192, 193, 197, 198, 199, 200, 201, 205, 206, 207, 208, 209, 211, 212, 213, 214, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 229, 230, 231, 232, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 2

Ranker:

. Popularity baseline:

In [6]:



def popularity_ranker(train_df):
    movie_counts = (
        train_df
        .groupby("item_id")
        .size()
        .reset_index(name="interaction_count")
        .sort_values(by="interaction_count", ascending=False)
    )
    return movie_counts

popularity_df = popularity_ranker(train)
popularity_m = popularity_ranker(train.head(M))

print(popularity_m)

print(popularity_df.head())

      item_id  interaction_count
249       258                 62
277       288                 59
95        100                 59
47         50                 58
174       181                 57
...       ...                ...
935      1026                  1
931      1022                  1
926      1017                  1
139       146                  1
1226     1680                  1

[1227 rows x 2 columns]
     item_id  interaction_count
49        50                453
99       100                419
257      258                419
180      181                405
285      286                384


. MF-SGD:

In [7]:

d = 20
lam = 0.05



rng = np.random.default_rng(0)
P = 0.1 * rng.standard_normal((n_users, d)).astype(np.float32)
Q = 0.1 * rng.standard_normal((n_items, d)).astype(np.float32)
bu = np.zeros(n_users, dtype=np.float32)
bi = np.zeros(n_items, dtype=np.float32)

print("Initialized P,Q,bu,bi")

Initialized P,Q,bu,bi


In [8]:
def rank_sgv(u_raw, items_raw, mu, bu, bi, P, Q, user2idx, item2idx):
    if u_raw not in user2idx:
        return []

    u = user2idx[u_raw]
    scores = []

    for i_raw in items_raw:
        if i_raw not in item2idx:
            continue

        i = item2idx[i_raw]
        score = mu + bu[u] + bi[i] + P[u] @ Q[i]
        scores.append((i_raw, score))

    return [i for i, _ in sorted(scores, key=lambda x: -x[1])]



mu = train_r.mean()
movies_sgv = rank_sgv(user, ratings['item_id'].unique(), mu, bu, bi, P, Q, user2idx, item2idx)

print(movies_sgv)

[93, 1353, 148, 861, 1022, 336, 1639, 55, 639, 301, 1262, 107, 21, 613, 988, 703, 405, 135, 1198, 1211, 480, 749, 335, 970, 270, 1032, 246, 1066, 159, 58, 713, 80, 650, 705, 671, 1238, 1674, 707, 368, 962, 484, 852, 1273, 1423, 629, 1458, 939, 1655, 212, 1026, 1292, 1135, 131, 1350, 960, 1650, 1190, 1542, 1062, 282, 701, 283, 1397, 137, 1411, 937, 780, 927, 398, 110, 296, 864, 1390, 1431, 205, 802, 1300, 1134, 1382, 945, 1192, 605, 524, 1439, 1392, 324, 42, 1475, 444, 783, 1176, 383, 587, 52, 1541, 1106, 571, 355, 1499, 513, 422, 541, 458, 575, 326, 891, 1628, 617, 1303, 537, 973, 648, 800, 585, 931, 1121, 974, 1054, 1681, 1120, 1666, 1489, 1302, 574, 1373, 512, 357, 146, 1137, 412, 1075, 1123, 679, 1047, 1015, 1521, 237, 667, 1563, 1313, 1412, 191, 1029, 561, 1369, 1169, 1580, 1409, 124, 171, 1534, 556, 1046, 820, 1365, 1493, 1019, 1653, 1027, 696, 1658, 1638, 645, 1084, 426, 329, 918, 1591, 1216, 716, 1581, 1143, 822, 823, 1492, 638, 1293, 1153, 240, 658, 519, 394, 1274, 127, 168, 20